In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import datetime
import pandas as pd
import matplotlib as mpl

sys.path.insert(0,"../src/")
import common.d2d as d2d

# from run_selection_single_channel import RunInfo
# import WaveformProcessor
# import FitSPE

# from common.utils import vec_regex_search

%run /home/ws/sk6801/sw/UCSD_analysis/SandyAQ/python_wrappers/notebooks/plot_style_kalinka.py



In [ ]:
def create_color_scheme(color_map: str, array: object, color_range=(0,1), darken=1, reverse=False):
    values = sorted(np.unique(array))
    if reverse:
        values = values[::-1]
        
    cmap = plt.get_cmap(color_map)
    color = cmap(np.linspace(color_range[0], color_range[1], len(values)))
    
    # https://stackoverflow.com/questions/37517587/how-can-i-change-the-intensity-of-a-colormap-in-matplotlib
    color[:,0:3] *= darken
    
    clr = {values[i]: color[i] for i in range(len(values))}
    return clr

In [ ]:
# before
# path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_dataplotsgain_info_single_channel_20240920.csv"

# after changing the SPE threshold
# path_GXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250608_all_gain_info_single_channel.csv"
path_GXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250812_all_gain_info_single_channel.csv"
# path_LXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250609_LXe_gain_info_single_channel.csv"
# path_LXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250610_LXe_gain_info_single_channel.csv"
# path_LXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250616_LXe_gain_info_single_channel.csv"
# path_LXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250706_LXe_gain_info_single_channel.csv"
path_LXe = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250812_LXe_gain_info_single_channel.csv"

df_GXe = pd.read_csv(path_GXe, 
                 parse_dates=["date_time"],
                 delimiter=",",
                 quotechar='"', 
                 skipinitialspace=True, 
                 encoding="utf-8")

df_LXe = pd.read_csv(path_LXe, 
                 parse_dates=["date_time"],
                 delimiter=",",
                 quotechar='"', 
                 skipinitialspace=True, 
                 encoding="utf-8")


df_all = pd.concat([df_GXe, df_LXe], axis=0)   


In [ ]:
class GainAnalysis:
    def __init__(self, df, output_path=None):
        self.df = df
        self.info = d2d.data(df)
        self.output_path = output_path
        
        self.settings()
        self.data_selection()
        self.voltage_calibration(calibration = False)
        self.create_dataframe()
        self.calculate_breakdown_voltage(output_path=self.output_path)


    def settings(self):
        self.color_temperature = create_color_scheme(
            "coolwarm_r", 
            self.info.temperature_K,
            darken = 0.8,
            color_range=(0.2,0.9),
            reverse=True)
        
        self.color_channel = create_color_scheme("viridis", 
                                    self.info.channel)

        self.color_voltage = create_color_scheme("hot", 
                                    self.info.voltage_preamp1_V,
                                    color_range=(0,0.8))

        self.dict_preamp_channel = {1: [0,1,2,3,4],
                       2: [5,6,7,8,10],
                       3: [9,11,12,13,14],
                       4: [15,16,17,18,19],
                       5: [20,21,22,23]}
        
        self.date_power_supply_changed = np.datetime64('2024-08-13')
        
    def data_selection(self):
        mask = (self.info.voltage_preamp1_V < -46) & (self.info.baseline_std_V < 0.025) & ~np.isnan(self.info.gain)
        self.info = self.info.apply_mask(mask)
        print("1: ", len(self.info))

        # remove files
        path = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_new/20241030_all_1_T98_all_voltages_6.0sig/meta_config_all_20241030_170524.json"
        mask = ~(self.info.md_full_path == path)
        self.info = self.info.apply_mask(mask)

        paths = ['/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_new/20241030_all_1_T98_all_voltages_6.0sig/meta_config_all_20241030_171835.json',
            '/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/raw_data/202410_LXe_tritium/20241031_all_1_T98_all_voltages_6.0sig/meta_config_all_20241031_113258.json']

        for path in paths:
            mask = ~((self.info.md_full_path==path))
            self.info = self.info.apply_mask(mask)

    def voltage_calibration(self, calibration: bool):
        if calibration == True:

            preamp_boards = np.arange(1,6)

            info_corrected = self.info.copy()
            info_corrected.voltage_preamp1_V = info_corrected.voltage_preamp1_V.astype(dtype=float)

            mask_time = (info_corrected.date_time > self.date_power_supply_changed)

            for i, set_voltage in enumerate(df_bias_V_correction_avg["set"]):
                select_voltage = info_corrected.voltage_preamp1_V == set_voltage
                
                for preamp_board in preamp_boards:
                    
                    in_board_mask = np.zeros(len(info_corrected), dtype=bool)
                    for channel in self.dict_preamp_channel[preamp_board]:
                        in_board = info_corrected.channel == channel
                        in_board_mask = in_board_mask | in_board
                    
                    mask = select_voltage & in_board_mask & mask_time
                    
                    corrected_voltage = df_bias_V_correction_avg.iloc[i][f"meas{preamp_board}"]
                    
                    # modify the voltage according to preamp board and set voltage
                    info_corrected.voltage_preamp1_V[mask] = corrected_voltage

            # # also get the df of into_corrected
            # info_corrected_df = info_corrected.get_df()
            # info_corrected_df["breakdown_voltage_V"] = pd.Series(dtype='float')
            # info_corrected_df["over_voltage_V"] = pd.Series(dtype='float')

        else: 
            self.info_corrected = self.info.copy()

    def create_dataframe(self):

        self.info_corrected_df = self.info_corrected.get_df().copy()
        
        # sort again and reset index
        self.info_corrected_df.sort_values("date_time", inplace=True)
        self.info_corrected_df.reset_index(drop=True, inplace=True)
        self.info_corrected_df["run_id"] = self.info_corrected_df.index + int(1)

        # setup columns for results
        self.info_corrected_df["breakdown_voltage_V"] = pd.Series(dtype='float')
        self.info_corrected_df["over_voltage_V"] = pd.Series(dtype='float')
        self.info_corrected_df["junction_capacity"] = pd.Series(dtype='float')
        self.info_corrected_df["total_capacity"] = pd.Series(dtype='float')
        self.info_corrected_df["date_str"] = pd.Series(dtype='float')
        
        # cluster data by date
        self.info_corrected_df["date_str"] = self.info_corrected_df["date_time_str"].str[:7]
        # print(np.unique(self.info_corrected_df.date_str))

        # overwrite the info_corrected with the new df
        self.info_corrected = d2d.data(self.info_corrected_df)

    def calculate_breakdown_voltage(self, output_path):

        if self.info_corrected is None:
            raise ValueError("info_corrected is not set. Please run the data selection and voltage calibration first.")
    
        # select data before and after change of power supply
        mask = self.info_corrected.date_time < self.date_power_supply_changed
        masked_before = self.info_corrected.apply_mask(mask)

        mask = self.info_corrected.date_time > self.date_power_supply_changed
        masked_after = self.info_corrected.apply_mask(mask)

        selected_data = [masked_before, masked_after]

        # temperatures
        temperature = np.unique(self.info_corrected.temperature_K)

        # loop through data sets and calculate breakdown voltage based on the date_cluster
        for data_set in selected_data:
            for tempe in temperature:
                mask = data_set.temperature_K == tempe
                masked_temp = data_set.apply_mask(mask)

                for i, channel in enumerate(np.unique(masked_temp.channel)):
                    mask = masked_temp.channel == channel
                    masked_channel = masked_temp.apply_mask(mask)
                    
                    for date_dataset in np.sort(np.unique(masked_channel.date_str)):
                        mask = masked_channel.date_str == date_dataset
                        masked_date = masked_channel.apply_mask(mask)

                        voltage_list = np.unique(masked_date.voltage_preamp1_V)
                        # n_points.append(len(voltage_list))
                        # date_str_list.append(date_dataset)
                        
                        if len(voltage_list) >= 3:
                            # print("Voltage points: ", voltage_list)

                            # fit
                            if masked_date.voltage_preamp1_V[0] < 0:
                                coef, res, _, _, _ = np.polyfit(-masked_date.voltage_preamp1_V,masked_date.gain,1, full=True)
                            else:
                                coef, res, _, _, _ = np.polyfit(masked_date.voltage_preamp1_V,masked_date.gain,1, full=True)

                            # print("Date: ", date_dataset, " Channel: ", channel, " Temperature: ", tempe, " Coefficients: ", coef)
                            # print("Length of filtered data: " , len(masked_date))

                            breakdown_voltage = -coef[1]/coef[0]
                            # print("Breakdown voltage: ", breakdown_voltage, " for date: ", date_dataset, " channel: ", channel, " temperature: ", tempe)


                            if breakdown_voltage > 0:
                                run_id_list = masked_date.run_id
                                # length += len(run_id_list)
                                

                                for id in run_id_list:
                                    path = self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "md_full_path"].values
                                    assert path in masked_date.md_full_path
                                    abs_bias_voltage = abs(self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "voltage_preamp1_V"])
                                    self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "breakdown_voltage_V"] = breakdown_voltage
                                    self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "over_voltage_V"] = abs_bias_voltage - breakdown_voltage
                                    self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "junction_capacity"] = -coef[0]*1.6e-19/33  # in F
                                    self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "total_capacity"] = -coef[0]*1.6e-19  # in F
                                    # self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "coef0"] = coef[0]
                                    # self.info_corrected_df.loc[self.info_corrected_df["run_id"]==id, "coef1"] = coef[1]

                                
                            
                            
                            # print("Number of runs: ", len(run_id_list))
                            # print("Breakdown voltage: ", breakdown_voltage, " for date: ", date_dataset, " channel: ", channel, " temperature: ", tempe)

                        # else:
                        #     print("Not enough voltage points for date: ", date_dataset, " channel: ", channel, " temperature: ", tempe)

        if output_path is not None:
            self.info_corrected_df.to_csv(output_path, sep=',', index=False, mode='w')
        else:
            print("Output path is not set, not saving the results.")
    
        # print(np.unique(self.info_corrected_df.date_str))
        self.info_corrected_Vbd = d2d.data(self.info_corrected_df)

        mask = (self.info_corrected_Vbd.breakdown_voltage_V > 0)
        # mask = ~np.isnan(self.info_corrected_Vbd.gain) & (self.info_corrected_Vbd.breakdown_voltage_V > 0)
        self.info_corrected_Vbd = self.info_corrected_Vbd.apply_mask(mask)
        # print("6: ", len(self.info_corrected_Vbd))
        # print(np.unique(self.info_corrected_Vbd.date_str))
        

### Load Data

In [ ]:
result_all = GainAnalysis(df_all, output_path = None)
result_all_df = result_all.info_corrected_Vbd.get_df()
info_corrected_Vbd = result_all.info_corrected_Vbd



### Finalized Plots

#### all data

In [ ]:
fig,ax = plt.subplots(figsize=(10,6))


temperature = np.unique(info_corrected_Vbd.temperature_K)

mask = ~np.isnan(info_corrected_Vbd.gain) & (info_corrected_Vbd.breakdown_voltage_V > 0)
gain_info = info_corrected_Vbd.apply_mask(mask)


# for i, channel in enumerate(np.unique(gain_info.channel)):
channel = 0
    
mask = (gain_info.channel == channel)
channel_info = gain_info.apply_mask(mask)

for tempe in temperature:

    # print(temperature)
    mask = channel_info.temperature_K == tempe
    masked_temp = channel_info.apply_mask(mask)
    
    y_list = []
    x_list = []
    run_id_list = []
    
    for j in range(len(masked_temp)):
        
        #gain = masked_temp.gain.mean()
        y_list.append(masked_temp.spe_resolution)
        
        x_list.append(masked_temp.gain/33)

    over_voltage = np.array(x_list).mean()
    gain = np.array(y_list).mean()

    # if i == 0:
    plt.plot(x_list, y_list, 
            "o-", 
            color=result_all.color_temperature[tempe],
            markersize=1
            )
            

# add items to legend
for temp in temperature:
    plt.scatter([], [], label=f"this work: {temp} K", 
        color=result_all.color_temperature[temp],
        marker='o',
        s=10)

plt.legend(bbox_to_anchor = (1,1.1), ncol=1)
# plt.gca().invert_xaxis()
plt.title(f"")
plt.ylabel("SPE resolution")
plt.xlabel("Over Voltage [V]")

# plt.ylim(0, 0.3)

# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/gain_over_voltage_v2.pdf", dpi=100, bbox_inches='tight')
# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/gain_over_voltage_v2.png")


#### Close up between 0 and 0.3

In [ ]:
fig,ax = plt.subplots(figsize=(10,6))


temperature = np.unique(info_corrected_Vbd.temperature_K)

mask = ~np.isnan(info_corrected_Vbd.gain) & (info_corrected_Vbd.breakdown_voltage_V > 0)
gain_info = info_corrected_Vbd.apply_mask(mask)


# for i, channel in enumerate(np.unique(gain_info.channel)):
channel = 0
    
mask = (gain_info.channel == channel)
channel_info = gain_info.apply_mask(mask)

for tempe in temperature:

    # print(temperature)
    mask = channel_info.temperature_K == tempe
    masked_temp = channel_info.apply_mask(mask)
    
    y_list = []
    x_list = []
    run_id_list = []
    
    for j in range(len(masked_temp)):
        
        #gain = masked_temp.gain.mean()
        y_list.append(masked_temp.spe_resolution)
        
        x_list.append(masked_temp.gain/33)

    over_voltage = np.array(x_list).mean()
    gain = np.array(y_list).mean()

    # if i == 0:
    plt.plot(x_list, y_list, 
            "o-", 
            color=result_all.color_temperature[tempe],
            markersize=1
            )
            

# add items to legend
for temp in temperature:
    plt.scatter([], [], label=f"this work: {temp} K", 
        color=result_all.color_temperature[temp],
        marker='o',
        s=10)

plt.legend(bbox_to_anchor = (1,1.1), ncol=1)
# plt.gca().invert_xaxis()
plt.title(f"")
plt.ylabel("SPE resolution")
plt.xlabel("Over Voltage [V]")

plt.ylim(0, 0.3)

# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/gain_over_voltage_v2.pdf", dpi=100, bbox_inches='tight')
# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/gain_over_voltage_v2.png")


##### Double check the gain correlation of this dataset

In [ ]:
fig,ax = plt.subplots(figsize=(10,6))

df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/nEXO_2022_overvoltage.csv")

voltage_col_name = df_lit.columns[::2]
gain_col_name = df_lit.columns[1::2]

color_list = ["#ff7f00", "#984ea3"]

for i, col_name in enumerate(voltage_col_name):
    temperature_label = col_name[:4]
    bias_voltage = df_lit[voltage_col_name[i]]
    gain = 1e3 * df_lit[gain_col_name[i]]
    ax.plot(bias_voltage, 
            gain,
            "o-",
            label = f"nEXO2022: {temperature_label}",
                    zorder=10,
                    color=color_list[i]
            )
    
    
df_lit = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/lit_values/baudis_2023_overvoltage.csv")

voltage_col_name = df_lit.columns[::2]
gain_col_name = df_lit.columns[1::2]

for i, col_name in enumerate(voltage_col_name[0:1]):
    temperature_label = col_name[:4]
    bias_voltage = df_lit[voltage_col_name[i]]
    gain = df_lit[gain_col_name[i]] # baudis 2018
    ax.plot(bias_voltage, 
            gain,
            "o-",
            markersize = 10,
            label = f"Peres2023: {temperature_label}",
                zorder=10,
                alpha = 0.8,
                markeredgecolor="black",
                color = "yellow"
            )


temperature = np.unique(info_corrected_Vbd.temperature_K)

mask = ~np.isnan(info_corrected_Vbd.gain) & (info_corrected_Vbd.breakdown_voltage_V > 0)
gain_info = info_corrected_Vbd.apply_mask(mask)


for i, channel in enumerate(np.unique(gain_info.channel)):
    
    mask = (gain_info.channel == channel)
    channel_info = gain_info.apply_mask(mask)
    
    for tempe in temperature:

        # print(temperature)
        mask = channel_info.temperature_K == tempe
        masked_temp = channel_info.apply_mask(mask)
        
        y_list = []
        x_list = []
        run_id_list = []
        
        for j in range(len(masked_temp)):
            
            #gain = masked_temp.gain.mean()
            y_list.append(masked_temp.gain/33)
            
            x_list.append(masked_temp.over_voltage_V)

        over_voltage = np.array(x_list).mean()
        gain = np.array(y_list).mean()

        # if i == 0:
        plt.plot(x_list, y_list, 
                "o-", 
                color=result_all.color_temperature[tempe],
                markersize=1
                )
            
        # else:
        #     plt.plot(over_voltage_list, gain_list, 
        #             "o-", 
        #             color=color_temperature[temperature],
        #             
        # plt.errorbar(tmp.voltage_preamp1_V, tmp.gain, 
        #          yerr=tmp.gain_err, 
        #          label = f"{temperature} K", 
        #          fmt="o", 
        #          ecolor = "black", 
        #          capsize=3,
        #          color=result_all.color_temperature[tempe]
        #          )


# add items to legend
for temp in temperature:
    plt.scatter([], [], label=f"this work: {temp} K", 
        color=result_all.color_temperature[temp],
        marker='o',
        s=10)

plt.legend(bbox_to_anchor = (1,1.1), ncol=1)
# plt.gca().invert_xaxis()
plt.title(f"")
plt.ylabel("Gain")
plt.xlabel("Over Voltage [V]")

# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/gain_over_voltage_v2.pdf", dpi=100, bbox_inches='tight')
# # plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/gain_over_voltage_v2.png")
